In [ ]:
import os


os.environ['KAGGLE_USERNAME'] = "ferroaprianto"


os.environ['KAGGLE_KEY'] = "KGAT_6f12f7ee4cd32fc932fb5fc3ae2a76a2"


!kaggle datasets download -d kwentar/blur-dataset

Dataset URL: https://www.kaggle.com/datasets/kwentar/blur-dataset
License(s): CC0-1.0
100% 1.49G/1.49G [00:23<00:00, 67.3MB/s]



In [ ]:
import zipfile
with zipfile.ZipFile('blur-dataset.zip', 'r') as zip_ref:
    zip_ref.extractall('dataset_blur')

In [ ]:
import os
print(os.listdir('dataset_blur'))

['sharp', 'motion_blurred', 'defocused_blurred', 'blur_dataset_scaled']


In [ ]:
import os
import shutil

base_dir = 'dataset_blur'

blur_dir = os.path.join(base_dir, 'blur')
os.makedirs(blur_dir, exist_ok=True)


folder_defocused = os.path.join(base_dir, 'defocused_blurred')
folder_motion = os.path.join(base_dir, 'motion_blurred')


for folder in [folder_defocused, folder_motion]:
    if os.path.exists(folder):
        for file in os.listdir(folder):
            shutil.move(os.path.join(folder, file), os.path.join(blur_dir, file))
        os.rmdir(folder)

print("Selesai! Sekarang kategorinya hanya:", os.listdir(base_dir))

Selesai! Sekarang kategorinya hanya: ['sharp', 'blur', 'blur_dataset_scaled']


In [ ]:
import shutil

folder_ekstra = os.path.join(base_dir, 'blur_dataset_scaled')
if os.path.exists(folder_ekstra):
    shutil.rmtree(folder_ekstra)
    print("Folder ekstra berhasil dihapus!")

print("Kategori final:", os.listdir(base_dir))

Folder ekstra berhasil dihapus!
Kategori final: ['sharp', 'blur']


In [ ]:
import os
import shutil
import tensorflow as tf
from tensorflow.keras import layers, models

base_dir = 'dataset_blur'

folder_ekstra = os.path.join(base_dir, 'blur_dataset_scaled')
if os.path.exists(folder_ekstra):
    shutil.rmtree(folder_ekstra)

print("Kategori di dalam folder sekarang:", os.listdir(base_dir))

train_ds = tf.keras.utils.image_dataset_from_directory(
    base_dir, validation_split=0.2, subset="training", seed=123, image_size=(224, 224), batch_size=32)
val_ds = tf.keras.utils.image_dataset_from_directory(
    base_dir, validation_split=0.2, subset="validation", seed=123, image_size=(224, 224), batch_size=32)

class_names = train_ds.class_names
print("Kategori yang dikenali AI:", class_names)

model = models.Sequential([
  layers.InputLayer(shape=(224, 224, 3)),
  layers.RandomFlip("horizontal"),
  layers.RandomRotation(0.1),
  layers.Rescaling(1./255),
  layers.Conv2D(16, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Conv2D(32, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Conv2D(64, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Flatten(),
  layers.Dropout(0.5),
  layers.Dense(128, activation='relu'),
  layers.Dense(len(class_names))
])

model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])


print("Memulai proses training...")
model.fit(train_ds, validation_data=val_ds, epochs=10)
print("Training Selesai! Model sudah di-reset dan diperbarui.")

Kategori di dalam folder sekarang: ['sharp', 'blur']
Found 1050 files belonging to 2 classes.
Using 840 files for training.
Found 1050 files belonging to 2 classes.
Using 210 files for validation.
Kategori yang dikenali AI: ['blur', 'sharp']
Memulai proses training...
Epoch 1/10
27/27 ━━━━━━━━━━━━━━━━━━━━ 50s 2s/step - accuracy: 0.5548 - loss: 1.0386 - val_accuracy: 0.6952 - val_loss: 0.6434
Epoch 2/10
27/27 ━━━━━━━━━━━━━━━━━━━━ 83s 2s/step - accuracy: 0.6595 - loss: 0.6644 - val_accuracy: 0.6952 - val_loss: 0.6254
Epoch 3/10
27/27 ━━━━━━━━━━━━━━━━━━━━ 80s 2s/step - accuracy: 0.6607 - loss: 0.6464 - val_accuracy: 0.7429 - val_loss: 0.6168
Epoch 4/10
27/27 ━━━━━━━━━━━━━━━━━━━━ 84s 2s/step - accuracy: 0.7036 - loss: 0.5994 - val_accuracy: 0.8095 - val_loss: 0.4702
Epoch 5/10
27/27 ━━━━━━━━━━━━━━━━━━━━ 45s 2s/step - accuracy: 0.7179 - loss: 0.5584 - val_accuracy: 0.7333 - val_loss: 0.6095
Epoch 6/10
27/27 ━━━━━━━━━━━━━━━━━━━━ 48s 2s/step - accuracy: 0.7869 - loss: 0.4726 - val_accuracy: 0

In [ ]:
import numpy as np
from google.colab import files
from tensorflow.keras.preprocessing import image

print("Silakan upload gambar untuk dites:")
uploaded = files.upload()

for fn in uploaded.keys():

  img = image.load_img(fn, target_size=(224, 224))
  img_array = image.img_to_array(img)
  img_array = tf.expand_dims(img_array, 0)


  predictions = model.predict(img_array)
  score = tf.nn.softmax(predictions[0])

  nama_kelas = class_names[np.argmax(score)]
  persentase = 100 * np.max(score)

  print(f"\n=> Hasil: Gambar ini terdeteksi sebagai **{nama_kelas.upper()}**")
  print(f"=> Tingkat keyakinan AI: {persentase:.2f}%")

Silakan upload gambar untuk dites:


Saving goole gole hd hd.jpeg to goole gole hd hd.jpeg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 136ms/step

=> Hasil: Gambar ini terdeteksi sebagai **BLUR**
=> Tingkat keyakinan AI: 93.44%
